In [19]:
import os
from  dotenv import load_dotenv
load_dotenv()
from langchain_community.document_loaders import PyMuPDFLoader

In [20]:
attention_file_path = r"doc\attention.pdf"  #- scan pdf not working

In [21]:
from langchain_community.document_loaders.parsers import RapidOCRBlobParser
loader = PyMuPDFLoader(
    file_path=attention_file_path,
    mode="page",
    extract_tables="markdown",
    extract_images=True,
    images_parser=RapidOCRBlobParser(),
    images_inner_format="markdown-img",
)

docs = loader.load()

In [22]:
import fitz  #
from langchain_core.documents import Document

In [23]:
doc = fitz.open(attention_file_path)
image_dir = "images"
os.makedirs(image_dir, exist_ok=True)
image_docs = []

for page_index in range(len(doc)):
    page = doc[page_index]
    image_list = page.get_images()

    for image_index, img in enumerate(image_list, start=1):
        xref = img[0]
        pix = fitz.Pixmap(doc, xref)

        if pix.n - pix.alpha > 3:
            pix = fitz.Pixmap(fitz.csRGB, pix)

        img_path = f"{image_dir}/page_{page_index+1}_img_{image_index}.png"
        pix.save(img_path)

        # Store image reference as document
        image_docs.append(
            Document(
                page_content=f"Image on page {page_index+1}: {img_path}",
                metadata={"type": "image"}
            )
        )